# RadLE Medical Custom Runtime

Experimental runner for `medgemma_1_5_4b`, `llava_med_mistral_7b`, and `internvl3_5_8b`. Run one selected model at a time on a GPU-backed Colab custom runtime. Outputs stay in Google Drive.

In [ ]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_medical_custom_runtime.py"
github_token = None

drive.mount("/content/drive")


def run_command(args, check=True, env=None):
    result = subprocess.run(args, text=True, capture_output=True, env=env)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {args[:3]}")
    return result


if (REPO_DIR / ".git").exists():
    pull_result = run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    if pull_result.returncode != 0:
        print("WARNING: git pull failed; continuing with the existing Colab checkout.")
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )

    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone the medical runtime module."
        )

    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
    run_command(["git", "clone", clone_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Medical runtime module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
# ==========================================
# 2. PYTHON DEPENDENCIES + CACHE ROUTING
# ==========================================
# Set INSTALL_SERVER_PACKAGES=False if your custom runtime image already has vLLM/SGLang.
INSTALL_SERVER_PACKAGES = True
SERVER_ENGINE = "vllm"  # "vllm" or "sglang"

# The GCP custom runtime often has a tight root disk and a larger empty /content disk.
# Put pip, Hugging Face, Transformers, and vLLM caches under /content before installing.
RUNTIME_CACHE_ROOT = pathlib.Path("/content/radle_runtime_cache")
cache_paths = {
    "HF_HOME": RUNTIME_CACHE_ROOT / "huggingface",
    "HUGGINGFACE_HUB_CACHE": RUNTIME_CACHE_ROOT / "huggingface" / "hub",
    "TRANSFORMERS_CACHE": RUNTIME_CACHE_ROOT / "huggingface" / "transformers",
    "PIP_CACHE_DIR": RUNTIME_CACHE_ROOT / "pip",
    "VLLM_CACHE_ROOT": RUNTIME_CACHE_ROOT / "vllm",
    "XDG_CACHE_HOME": RUNTIME_CACHE_ROOT / "xdg",
}
for key, path in cache_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)

print("Cache root:", RUNTIME_CACHE_ROOT)
run_command(["df", "-h", "/", "/content"], check=False)
run_command(["nvidia-smi"], check=False)

base_packages = [
    "openai",
    "pandas",
    "anthropic",
    "google-genai",
    "huggingface_hub",
]
run_command([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "--cache-dir",
    os.environ["PIP_CACHE_DIR"],
    *base_packages,
])

if INSTALL_SERVER_PACKAGES:
    if SERVER_ENGINE == "vllm":
        run_command([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--cache-dir",
            os.environ["PIP_CACHE_DIR"],
            "vllm",
        ])
    elif SERVER_ENGINE == "sglang":
        run_command([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--cache-dir",
            os.environ["PIP_CACHE_DIR"],
            "sglang[all]",
        ])
    else:
        raise ValueError("SERVER_ENGINE must be 'vllm' or 'sglang'.")


In [ ]:
# ==========================================
# 3. IMPORT MEDICAL RUNTIME HELPERS
# ==========================================
import importlib

import radle_benchmark
import radle_medical_custom_runtime as medical_runtime

radle_benchmark = importlib.reload(radle_benchmark)
medical_runtime = importlib.reload(medical_runtime)
medical_runtime.configure_cache_environment(str(RUNTIME_CACHE_ROOT))

print("Benchmark module:", radle_benchmark.__file__)
print("Medical runtime module:", medical_runtime.__file__)
print("Supported medical models:")
medical_runtime.print_model_roster()
medical_runtime.print_runtime_resources()


In [ ]:
# ==========================================
# 4. MODEL, SERVER, AND RUN CONFIG
# ==========================================
from pathlib import Path

# Recommended sequence: run one model with TEST_LIMIT=1, then 3, then 5.
SELECTED_MODEL_NAME = "medgemma_1_5_4b"
# SELECTED_MODEL_NAME = "llava_med_mistral_7b"
# SELECTED_MODEL_NAME = "internvl3_5_8b"

TEST_LIMIT = 1
RUN_LABEL = "medical_test_1_case"
RESUME = True

# Drive dataset root matches the official RadLE notebook.
dataset_root = Path("/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset")
run_paths = medical_runtime.build_medical_run_paths(
    dataset_root,
    model_name=SELECTED_MODEL_NAME,
    run_label=RUN_LABEL,
)

master_images_folder = run_paths["master_images_folder"]
raw_results_csv = run_paths["raw_results_csv"]
raw_backup_dir = run_paths["raw_backup_dir"]
scorer_csv = run_paths["scorer_view_csv"]

# Local server config. Use START_SERVER=False if you already started an OpenAI-compatible endpoint.
START_SERVER = True
HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}/v1"
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 8192
SERVER_TIMEOUT_SECONDS = 900
EXTRA_SERVER_ARGS = []

print("Selected model:", SELECTED_MODEL_NAME)
print("Run folder:", run_paths["run_root"])
print("Raw results CSV:", raw_results_csv)
print("Endpoint:", BASE_URL)


In [ ]:
# ==========================================
# 5. HUGGING FACE ACCESS FOR GATED MODELS
# ==========================================
# MedGemma requires accepting the model terms on Hugging Face and setting HF_TOKEN.
hf_token = None
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets; value hidden.")
else:
    print("No HF_TOKEN secret found. This is OK for ungated models, but MedGemma needs it.")


In [ ]:
# ==========================================
# 6. START OR CONNECT TO LOCAL OPENAI-COMPATIBLE SERVER
# ==========================================
server_process = None
server_log_path = f"/content/{SELECTED_MODEL_NAME}_{SERVER_ENGINE}_server.log"

if START_SERVER:
    print(f"Starting {SERVER_ENGINE} server for {SELECTED_MODEL_NAME}...")
    server_process = medical_runtime.start_model_server(
        model_name=SELECTED_MODEL_NAME,
        engine=SERVER_ENGINE,
        host=HOST,
        port=PORT,
        hf_token=hf_token,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        extra_args=EXTRA_SERVER_ARGS,
        log_path=server_log_path,
    )
    print("Server process started. Log path:", server_log_path)
    print("Model download/load can take several minutes on the first run.")
else:
    print("START_SERVER=False; connecting to existing endpoint.")

try:
    models_payload = medical_runtime.wait_for_openai_server(
        base_url=BASE_URL,
        timeout_seconds=SERVER_TIMEOUT_SECONDS,
    )
except Exception:
    print("Server did not become ready. Last server log lines:")
    print(medical_runtime.read_log_tail(server_log_path, lines=120))
    raise

print("Endpoint /models response:")
print(models_payload)

endpoint_client = medical_runtime.make_openai_client(BASE_URL)


In [ ]:
# ==========================================
# 7. RUN ONE-MODEL MEDICAL RADLE SMOKE
# ==========================================
from IPython.display import display

df_final = medical_runtime.run_medical_model_benchmark(
    client=endpoint_client,
    model_name=SELECTED_MODEL_NAME,
    image_folder=master_images_folder,
    output_csv=raw_results_csv,
    test_limit=TEST_LIMIT,
    backup_dir=raw_backup_dir,
    resume=RESUME,
)

rows = medical_runtime.validate_output_csv(raw_results_csv, SELECTED_MODEL_NAME)
print(f"Validated output CSV rows: {rows}")
display(df_final.head())


In [ ]:
# ==========================================
# 8. SCORER VIEW + READ-ONLY AUDIT
# ==========================================
active_model = [medical_runtime.get_model_config(SELECTED_MODEL_NAME)]

df_scorer, display_df, scorer_csv = radle_benchmark.create_scorer_view(
    raw_results_csv,
    scorer_csv=scorer_csv,
)
print("Scorer CSV:", scorer_csv)
display(display_df)

audit_results = radle_benchmark.audit_benchmark_output(
    raw_csv=raw_results_csv,
    models=active_model,
    expected_case_ids=None,
)

print("DATASET INTEGRITY:")
display(audit_results["dataset_integrity"])
print("BUCKET SUMMARY:")
display(audit_results["bucket_summary"])
print("REPAIR TARGETS:")
display(audit_results["repair_targets"].head(50))


In [ ]:
# ==========================================
# 9. STOP LOCAL SERVER WHEN DONE
# ==========================================
# Run this cell before switching SELECTED_MODEL_NAME, or restart the runtime.
medical_runtime.stop_model_server(server_process)
print("Server stopped if it was started by this notebook.")
